# QAOA with native FlagQuantum quantum functions

Complete both original sections: four-layer MaxCut and three-layer constrained
Knapsack. Reuse native phase and mixer helpers, prepare a uniform state, and
synthesize with explicit budgets. Parameters are radians. No foreign SDK is used.

MaxCut has edges (0,1), (0,2), (1,2), (1,3), (2,4), (3,4). Its cost is the
negative cut count divided by six. Knapsack has a in [0,7], b in [0,3], value
3a+5b and feasibility 2a+3b <= 12; infeasible objective is zero.


In [ ]:
import matplotlib.pyplot as plt

import flagquantum as fq
from examples.synthesis.algorithms.search_and_optimization.qaoa.main import (
    ansatz as knapsack_layers,
)
from examples.synthesis.algorithms.search_and_optimization.qaoa.overview.plots import (
    knapsack_figure,
    maxcut_figure,
)
from examples.synthesis.algorithms.search_and_optimization.qaoa.overview.workflow import (
    classical_cost,
    initial_parameters,
    run_workflow,
)
from examples.synthesis.algorithms.search_and_optimization.qaoa.qaoa_max_cut.main import (
    maxcut_cost_layer,
    mixer_layer,
)
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QArray,
    QBit,
    QNum,
    allocate,
    hadamard_transform,
    qfunc,
)


@qfunc
def maxcut_program(params: list[float], v: Output[QArray[QBit]]) -> None:
    allocate(5, v)
    hadamard_transform(v)
    for layer in range(4):
        maxcut_cost_layer(params[layer], v)
        mixer_layer(params[4 + layer], v)


@qfunc
def knapsack_program(params: list[float], a: Output[QNum], b: Output[QNum]) -> None:
    allocate(3, a)
    allocate(2, b)
    hadamard_transform(a)
    hadamard_transform(b)
    knapsack_layers(params, a, b)


## Inspect the initial native programs

Keep the source linear schedules: four MaxCut gammas 0..1 then four betas 1..0;
three Knapsack gammas 0..pi then three betas pi..0. Named numeric readout avoids
physical wire numbers and bitstring decoding. Array node i is named bit i.


In [ ]:
maxcut_initial = synth.synthesize(
    maxcut_program, classical_inputs={"params": initial_parameters("maxcut")},
    max_qubits=5, max_workspace=0, max_operations=2000,
    memory_limit_bytes=4 * 1024**2,
)
knapsack_initial = synth.synthesize(
    knapsack_program, classical_inputs={"params": initial_parameters("knapsack")},
    max_qubits=5, max_workspace=0, max_operations=2000,
    memory_limit_bytes=4 * 1024**2,
)
print("MaxCut", dict(maxcut_initial.probabilities(fq.run(maxcut_initial), registers=("v",))))
print("Knapsack", dict(knapsack_initial.probabilities(fq.run(knapsack_initial), registers=("a", "b"))))
print("MaxCut resources", maxcut_initial.report["resources"])
print("Knapsack resources", knapsack_initial.report["resources"])


## Optimize MaxCut and inspect actual sampled cuts

Every evaluation synthesizes and executes the retained native quantum functions.
Public native Nelder-Mead replaces source COBYLA; exact local expected cost
replaces shot estimation during optimization. Preserve the source 60-iteration
budget, disclose iteration/evaluation counts and report nonconvergence honestly.
Final counts are 1000 actual CPU samples with seed 2718. These replacements do
not imply an identical optimizer trajectory or a quantum advantage.


In [ ]:
maxcut_workflow = run_workflow("maxcut")
print("initial cost", maxcut_workflow.trace[0].cost)
print("final expected cost", maxcut_workflow.optimizer.value)
print("iterations", maxcut_workflow.optimizer.iterations)
print("actual evaluations", maxcut_workflow.optimizer.evaluations)
print("converged", maxcut_workflow.optimizer.converged)
print("best observed feasible", maxcut_workflow.best_feasible)
for sample in maxcut_workflow.samples[:10]:
    print(sample)
maxcut_plot = maxcut_figure(maxcut_workflow)
plt.show()
plt.close(maxcut_plot)


## Optimize constrained Knapsack

Apply the coherent weight predicate in each of the three quantum layers.
Read a/b as integers; keep feasibility distinct from the objective. If no
feasible assignment is observed, report None rather than inserting an exact
classical answer. The plotted trace and table come from actual evaluations and
1000 final seeded samples, including the original <=12 boundary.


In [ ]:
knapsack_workflow = run_workflow("knapsack")
print("initial cost", knapsack_workflow.trace[0].cost)
print("final expected cost", knapsack_workflow.optimizer.value)
print("iterations", knapsack_workflow.optimizer.iterations)
print("actual evaluations", knapsack_workflow.optimizer.evaluations)
print("converged", knapsack_workflow.optimizer.converged)
print("best observed feasible", knapsack_workflow.best_feasible)
for sample in knapsack_workflow.samples[:10]:
    a, b = sample.assignment
    print("a", a, "b", b, "value", 3*a+5*b, "feasible", sample.feasible, "count", sample.count)
knapsack_plot = knapsack_figure(knapsack_workflow)
plt.show()
plt.close(knapsack_plot)


## Independent bounded reference after the search

The 32-assignment classical references below are labeled checks only. They do
not supply optimizer parameters, probabilities or sampled solutions. Seeing an
optimal assignment in a finite sample does not establish optimizer convergence.


In [ ]:
maxcut_exact_cost = min(classical_cost("maxcut", (word,))[0] for word in range(32))
knapsack_exact_cost = min(
    -3*a-5*b for a in range(8) for b in range(4) if 2*a+3*b <= 12
)
print("exact bounded MaxCut reference", maxcut_exact_cost)
print("exact bounded Knapsack reference", knapsack_exact_cost)
print("observed MaxCut", maxcut_workflow.best_feasible)
print("observed Knapsack", knapsack_workflow.best_feasible)
